# Tópico 5 e 6: Variável-Alvo e Feature Engineering
Este notebook é responsável por:
1. Criar a variável-alvo (`quebra_safra`) baseada no limiar de rendimento (Tópico 5).
2. Remover variáveis que causariam Target Leakage (Vazamento de Dados).
3. Criar novas variáveis preditoras baseadas nos dados agroclimáticos (Tópico 6).

In [1]:
import pandas as pd
import os

# Carregar os dados brutos integrados
df_raw = pd.read_csv('../data/raw/integrado.csv')
display(df_raw.head())

,ano,area_plantada_ha,area_colhida_ha,quantidade_produzida_t,rendimento_kg_ha,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,GWETROOT
0,2015,625900.0,619900.0,1951710.0,3148.0,645.49,27.850904,41.79,17.31,61.046274,0.635616
1,2016,620000.0,615000.0,1771200.0,2880.0,1147.75,26.832432,39.57,15.78,66.810902,0.658962
2,2017,620000.0,620000.0,2157600.0,3480.0,1561.12,25.576301,39.86,12.24,74.063096,0.714904
3,2018,600000.0,600000.0,2232000.0,3720.0,1903.15,25.089068,39.71,13.42,77.658548,0.755589
4,2019,605000.0,605000.0,2141700.0,3540.0,1210.35,26.609890,40.91,15.47,68.237671,0.675699


## Tópico 5: Definição da Variável-Alvo
- **Alvo (`quebra_safra`)**: Definido como `1` se `rendimento_kg_ha < 2450`, e `0` caso contrário (conforme `RFC.md`).
- **Target Leakage**: Variáveis que representam informações de "pós-colheita" não podem ser usadas para prever o risco antes da colheita. Por isso, excluímos: `rendimento_kg_ha`, `quantidade_produzida_t`, `area_colhida_ha`, `area_plantada_ha`.

In [2]:
df_target = df_raw.copy()

# 1. Cria a variável alvo
df_target['quebra_safra'] = (df_target['rendimento_kg_ha'] < 2450).astype(int)

# 2. Remoção das variáveis vazadas
colunas_vazamento = [
    'rendimento_kg_ha', 
    'quantidade_produzida_t', 
    'area_colhida_ha', 
    'area_plantada_ha'
]
df_target = df_target.drop(columns=colunas_vazamento)

print("Colunas após o Tópico 5:")
print(df_target.columns.tolist())
display(df_target[['ano', 'quebra_safra']].head())

Colunas após o Tópico 5:
['ano', 'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT', 'quebra_safra']


,ano,quebra_safra
0,2015,0
1,2016,0
2,2017,0
3,2018,0
4,2019,0


## Tópico 6: Criação de Novas Variáveis (Feature Engineering)
Vamos criar variáveis meteorológicas derivadas que possam explicar melhor o estresse sofrido pela planta:
1. **Amplitude Térmica (`amplitude_termica`)**: Variações muito grandes de temperatura no mesmo dia geram estresse hídrico e fisiológico.
2. **Índice de Precipitação por Temperatura (`precipitacao_por_temperatura`)**: Mede o balanço de água disponível frente à evaporação gerada pelo calor.
3. **Interação Umidade x Precipitação (`GWET_x_PREC`)**: Identifica anos onde choveu muito e o solo efetivamente reteve essa água.

In [3]:
df_features = df_target.copy()

# 1. Amplitude Térmica Diária Média
df_features['amplitude_termica'] = df_features['T2M_MAX'] - df_features['T2M_MIN']

# 2. Indice de Estresse Hídrico Simples (Chuva / Temperatura Média)
df_features['precipitacao_por_temperatura'] = df_features['PRECTOTCORR'] / (df_features['T2M'] + 0.001)

# 3. Interação: Umidade do Solo x Precipitação
df_features['GWET_x_PREC'] = df_features['GWETROOT'] * df_features['PRECTOTCORR']

display(df_features.head())

,ano,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,GWETROOT,quebra_safra,amplitude_termica,precipitacao_por_temperatura,GWET_x_PREC
0,2015,645.49,27.850904,41.79,17.31,61.046274,0.635616,0,24.48,23.175794,410.284055
1,2016,1147.75,26.832432,39.57,15.78,66.810902,0.658962,0,23.79,42.773135,756.323347
2,2017,1561.12,25.576301,39.86,12.24,74.063096,0.714904,0,27.62,61.035368,1116.051104
3,2018,1903.15,25.089068,39.71,13.42,77.658548,0.755589,0,26.29,75.852722,1437.999284
4,2019,1210.35,26.609890,40.91,15.47,68.237671,0.675699,0,25.44,45.483258,817.831837


## Salvando os dados processados
Salvaremos na pasta `data/interim/` para que os próximos passos do projeto possam consumi-los.

In [4]:
# Criar diretório se não existir
os.makedirs('../data/interim', exist_ok=True)

# Salvar o dataset com features
path_out = '../data/interim/04_dados_com_features.csv'
df_features.to_csv(path_out, index=False)
print(f"Dados salvos com sucesso em: {path_out}")

Dados salvos com sucesso em: ../data/interim/04_dados_com_features.csv
